# 11 — Function Calling Training

Train the model to correctly invoke `aro ask` tools (function calling).
ARO Coder has 18 tools available via `aro ask`. This notebook generates
training pairs that teach the model:

1. **When** to call each tool (trigger recognition)
2. **How** to format the call (correct JSON arguments)
3. **What** to do with the result (interpret + next step)
4. **Fix patterns** — `aro_check` error → `edit_file` fix → verify

**Input:** Tool definitions from `../../Sources/AROAsk/Tools/`
**Output:** `../data/02_knowledge/knowledge_pairs.jsonl` (appended)

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')

import json, re, random
from pathlib import Path
from collections import Counter

DATA_OUT = DATA_ROOT / '11_function_calling'
DATA_OUT.mkdir(parents=True, exist_ok=True)
print(f'Output: {DATA_OUT}')

## Tool definitions — extracted from the real `aro ask` sources

The tool inventory is no longer a hardcoded copy (issue #397). `extract_ask_tools()`
in `config.py` parses every `AskToolDescriptor` in `Sources/AROAsk/Tools/` and
`Sources/AROAsk/Retrieval/` at pipeline runtime, so names and parameter schemas
always match what `aro ask` actually ships. The notebook **fails loudly** if the
extracted inventory is missing expected core tools or if a hand-curated example
uses a parameter that does not exist in the real schema. A hardcoded snapshot is
kept ONLY as a documented fallback when extraction itself fails (with a stderr
warning).

In [ ]:
# ── Tool inventory: extracted from the real `aro ask` Swift sources (#397) ───
# Source of truth: Sources/AROAsk/Tools/*.swift + Sources/AROAsk/Retrieval/.
# Schemas (names, params, required) come from extraction; only the example
# calls below are hand-curated — and they are validated against the extracted
# schemas so any drift fails the notebook instead of poisoning training data.
import sys as _sys

# Hand-curated example calls per tool. Keys must exist in the extracted
# inventory; args must be a subset of the tool's real parameters.
TOOL_EXAMPLES = {
    'read_file': [
        {'user': 'Read the main.aro file', 'args': {'path': 'main.aro'}},
        {'user': 'Show me lines 10-20 of users.aro', 'args': {'path': 'users.aro', 'offset': 10, 'limit': 10}},
    ],
    'write_file': [
        {'user': 'Create a hello world ARO app', 'args': {'path': 'main.aro', 'content': '(Application-Start: Hello) {\n    Log "Hello!" to the <console>.\n    Return an <OK: status> for the <greeting>.\n}'}},
    ],
    'edit_file': [
        {'user': 'Fix the string concat in main.aro', 'args': {'path': 'main.aro', 'old_string': '<first> + " " + <last>', 'new_string': '<first> ++ " " ++ <last>'}},
    ],
    'list_dir': [
        {'user': 'What files are in this project?', 'args': {}},
    ],
    'grep': [
        {'user': 'Find all feature sets that use Emit', 'args': {'pattern': 'Emit', 'glob': '*.aro'}},
    ],
    'run_shell': [
        {'user': 'Show the git log', 'args': {'command': 'git log --oneline -5'}},
    ],
    'aro_check': [
        {'user': 'Check if my app has syntax errors', 'args': {'path': '.'}},
        {'user': 'Validate the code', 'args': {'path': '.'}},
    ],
    'aro_run': [
        {'user': 'Run the app', 'args': {'path': '.'}},
    ],
    'aro_test': [
        {'user': 'Run the tests', 'args': {'path': '.'}},
    ],
    'aro_build': [
        {'user': 'Build a native binary', 'args': {'path': '.'}},
    ],
    'parse_aro': [
        {'user': 'Show me the AST', 'args': {'path': 'main.aro'}},
    ],
    'list_actions': [
        {'user': 'What actions can I use?', 'args': {}},
    ],
    'create_plugin': [
        {'user': 'Create a Swift plugin for analytics', 'args': {'name': 'analytics', 'language': 'swift', 'handle': 'Analytics'}},
    ],
    'write_openapi': [
        # Real schema takes title + paths (NOT a raw `content` string — that
        # was drift in the old hardcoded copy, caught by extraction).
        {'user': 'Generate an OpenAPI contract for my user API',
         'args': {'title': 'User API', 'paths': [
             {'path': '/users', 'method': 'get', 'operationId': 'listUsers', 'summary': 'List all users'},
             {'path': '/users', 'method': 'post', 'operationId': 'createUser', 'summary': 'Create a user'},
             {'path': '/users/{id}', 'method': 'get', 'operationId': 'getUser', 'summary': 'Get one user'},
         ]}},
    ],
    'generate_docs': [
        {'user': 'Generate docs for my app', 'args': {'path': '.'}},
    ],
    'list_proposals': [
        {'user': 'What language proposals exist?', 'args': {}},
    ],
    'read_proposal': [
        {'user': 'Show me the events proposal', 'args': {'number': '0007'}},
    ],
    'search_project': [
        {'user': 'Find code related to authentication', 'args': {'query': 'user authentication login'}},
    ],
}

# Core tools that MUST be present in the extracted inventory. If the sources
# rename or remove one of these, the notebook fails loudly so the training
# data is fixed together with the toolset.
EXPECTED_CORE_TOOLS = {
    'read_file', 'write_file', 'edit_file', 'list_dir', 'grep', 'run_shell',
    'aro_check', 'aro_run', 'aro_test', 'aro_build', 'parse_aro',
    'list_actions', 'create_plugin', 'write_openapi', 'generate_docs',
    'list_proposals', 'read_proposal', 'search_project',
}

# Documented fallback: last-known-good snapshot of extract_ask_tools().
# ONLY used when extraction itself fails (e.g. Sources/ not checked out).
FALLBACK_TOOL_SCHEMAS = [
    {'name': 'aro_build', 'params': ['path'], 'required': ['path']},
    {'name': 'aro_check', 'params': ['path'], 'required': ['path']},
    {'name': 'aro_run', 'params': ['path', 'args?'], 'required': ['path']},
    {'name': 'aro_test', 'params': ['path'], 'required': ['path']},
    {'name': 'create_plugin', 'params': ['name', 'language', 'handle', 'actions?', 'qualifiers?'], 'required': ['name', 'language', 'handle']},
    {'name': 'edit_file', 'params': ['path', 'old_string', 'new_string'], 'required': ['path', 'old_string', 'new_string']},
    {'name': 'generate_docs', 'params': ['path', 'output_path?'], 'required': ['path']},
    {'name': 'grep', 'params': ['pattern', 'path?', 'glob?'], 'required': ['pattern']},
    {'name': 'list_actions', 'params': [], 'required': []},
    {'name': 'list_dir', 'params': ['path?'], 'required': []},
    {'name': 'list_proposals', 'params': [], 'required': []},
    {'name': 'parse_aro', 'params': ['path'], 'required': ['path']},
    {'name': 'read_file', 'params': ['path', 'offset?', 'limit?'], 'required': ['path']},
    {'name': 'read_proposal', 'params': ['number'], 'required': ['number']},
    {'name': 'run_shell', 'params': ['command', 'timeout?'], 'required': ['command']},
    {'name': 'search_project', 'params': ['query', 'k?'], 'required': ['query']},
    {'name': 'write_file', 'params': ['path', 'content'], 'required': ['path', 'content']},
    {'name': 'write_openapi', 'params': ['title', 'paths', 'version?', 'output_path?'], 'required': ['title', 'paths']},
]

try:
    TOOL_SCHEMAS = extract_ask_tools(ARO_ROOT)
    _missing = EXPECTED_CORE_TOOLS - {t['name'] for t in TOOL_SCHEMAS}
    if _missing:
        raise RuntimeError(
            f'aro ask tool inventory drift — expected core tools missing '
            f'from Sources/: {sorted(_missing)}'
        )
    TOOLS_FROM_SOURCE = True
    print(f'Extracted {len(TOOL_SCHEMAS)} tools from Swift sources:')
    for t in TOOL_SCHEMAS:
        print(f"  {t['name']:<16} ({', '.join(t['params']) or 'no params'})  [{t['source_file']}]")
except Exception as _e:
    _sys.stderr.write(
        f'[NB11] WARNING: could not extract the aro ask tool inventory from '
        f'Sources/ ({_e}).\n[NB11] Falling back to the hardcoded snapshot — '
        f'training data may drift from the real toolset!\n'
    )
    TOOL_SCHEMAS = FALLBACK_TOOL_SCHEMAS
    TOOLS_FROM_SOURCE = False

TOOLS = [{
    'name':        t['name'],
    'params':      t['params'],
    'required':    t.get('required', []),
    'description': t.get('description', ''),
    'examples':    TOOL_EXAMPLES.get(t['name'], []),
} for t in TOOL_SCHEMAS]

# ── Drift guard: hand-curated examples must match the real schemas ───────────
_param_names = {t['name']: {p.rstrip('?') for p in t['params']} for t in TOOL_SCHEMAS}
for _name, _examples in TOOL_EXAMPLES.items():
    if _name not in _param_names:
        raise RuntimeError(
            f'TOOL_EXAMPLES contains {_name!r} which is not in the extracted '
            f'inventory — remove or update the example')
    for _ex in _examples:
        _unknown = set(_ex['args']) - _param_names[_name]
        if _unknown:
            raise RuntimeError(
                f'Example for {_name!r} uses unknown parameter(s) '
                f'{sorted(_unknown)} — the real schema is {sorted(_param_names[_name])}')

_no_examples = [t['name'] for t in TOOLS if not t['examples']]
if _no_examples:
    print(f'NOTE: extracted tools without hand-curated examples: {", ".join(_no_examples)}')
print(f"{len(TOOLS)} tools, {sum(len(t['examples']) for t in TOOLS)} examples | "
      f"schema source: {'Sources/AROAsk (live)' if TOOLS_FROM_SOURCE else 'hardcoded fallback'}")


## Generate training pairs

Three pair types:
1. **Direct call** — user asks → model calls the right tool
2. **Fix chains** — read → check → edit → verify (the `/fix` workflow)
3. **Tool selection** — given a task, which tool to use?

In [ ]:
import json as _json

def tool_call(name, args):
    return f'<tool_call>{_json.dumps({"name": name, "arguments": args})}</tool_call>'

pairs = []

# ---- Type 1: Direct tool calls ----
for tool in TOOLS:
    for ex in tool['examples']:
        pairs.append({
            'instruction': ex['user'],
            'output': tool_call(tool['name'], ex['args']),
            'source': f'fc_direct:{tool["name"]}',
            'task_type': 'function_calling',
            'score': 1.0,
        })
print(f'Direct calls: {len(pairs)}')

# ---- Type 2: Fix chains (aro ask /fix workflow) ----
FIX_SCENARIOS = [
    {'code': '(Application-Start: Demo) {\n    Compute the <msg> from "Hello" + <name>.\n    Log <msg> to the <console>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:45: error: Expected \'\'>\', but got +',
     'old': '"Hello" + <name>', 'new': '"Hello, " ++ <name>', 'desc': 'string concat uses ++'},
    {'code': '(Application-Start: Demo) {\n    Create the <count> with 0.\n    Compute the <count> from <count> + 1.\n    Return an <OK: status> for the <result>.\n}',
     'error': '3:18: error: Cannot rebind variable \'count\'',
     'old': 'Compute the <count>', 'new': 'Compute the <incremented>', 'desc': 'immutable variable'},
    {'code': '(Application-Start: Demo) {\n    Compute the <is-valid> from <age> > 18.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:22: error: Reserved prefix \'is-\'',
     'old': '<is-valid>', 'new': '<valid>', 'desc': 'reserved prefix'},
    {'code': '(Application-Start: Demo) {\n    Retrieve the <user> with the <user-repo>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:30: error: Expected preposition \'from\', but got \'with\'',
     'old': 'Retrieve the <user> with', 'new': 'Retrieve the <user> from', 'desc': 'wrong preposition'},
    {'code': '(Application-Start: Demo) {\n    Store the <user> from the <user-repo>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:24: error: Expected preposition \'into\', but got \'from\'',
     'old': 'Store the <user> from', 'new': 'Store the <user> into', 'desc': 'Store uses into'},
    {'code': '(Application-Start: Demo) {\n    Log <msg> for the <console>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:18: error: Expected preposition \'to\', but got \'for\'',
     'old': 'Log <msg> for the', 'new': 'Log <msg> to the', 'desc': 'Log uses to'},
    {'code': '(Application-Start: Demo) {\n    Emit the <user> to the <event>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:10: error: Expected event qualifier',
     'old': 'Emit the <user> to the <event>.', 'new': 'Emit a <UserCreated: event> with <user>.', 'desc': 'Emit syntax'},
    {'code': '(Application-Start: Demo) {\n    Extract the <id> from the <request: id>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:40: error: Use pathParameters for URL params',
     'old': '<request: id>', 'new': '<pathParameters: id>', 'desc': 'path params'},
]

for s in FIX_SCENARIOS:
    # Full chain: user asks to fix -> read -> check -> edit -> verify
    # Step 1: user asks, model reads file
    pairs.append({'instruction': 'Fix the errors in main.aro',
                  'output': tool_call('read_file', {'path': 'main.aro'}),
                  'source': 'fc_fix:read', 'task_type': 'function_calling', 'score': 1.0})
    # Step 2: model sees code, runs check
    pairs.append({'instruction': f'File content:\n```aro\n{s["code"]}\n```\nCheck for errors.',
                  'output': tool_call('aro_check', {'path': '.'}),
                  'source': 'fc_fix:check', 'task_type': 'function_calling', 'score': 1.0})
    # Step 3: model sees error, fixes with edit
    pairs.append({'instruction': f'aro_check error:\n{s["error"]}\nFix it.',
                  'output': tool_call('edit_file', {'path': 'main.aro', 'old_string': s['old'], 'new_string': s['new']}),
                  'source': 'fc_fix:edit', 'task_type': 'function_calling', 'score': 1.0})
    # Step 4: verify
    pairs.append({'instruction': f'Fixed {s["desc"]}. Verify.',
                  'output': tool_call('aro_check', {'path': '.'}),
                  'source': 'fc_fix:verify', 'task_type': 'function_calling', 'score': 1.0})

print(f'Fix chains: {len(FIX_SCENARIOS) * 4}')

# ---- Type 3: Tool selection (which tool for this task?) ----
TOOL_SELECTION = [
    ('I want to see what files exist', 'list_dir'),
    ('Check my code for errors', 'aro_check'),
    ('Run my application', 'aro_run'),
    ('Find where the user model is defined', 'grep'),
    ('Create a new Rust plugin', 'create_plugin'),
    ('Generate API documentation', 'generate_docs'),
    ('Build a native binary of my app', 'aro_build'),
    ('What ARO actions are available?', 'list_actions'),
    ('Read the control flow proposal', 'read_proposal'),
    ('Search for authentication-related code', 'search_project'),
    ('Run the test suite', 'aro_test'),
    ('Show me the parse tree', 'parse_aro'),
    ('Write an openapi.yaml for my API', 'write_openapi'),
    ('Replace the old function name with the new one', 'edit_file'),
    ('Execute a git command', 'run_shell'),
]
for user_q, tool_name in TOOL_SELECTION:
    tool = next(t for t in TOOLS if t['name'] == tool_name)
    args = tool['examples'][0]['args'] if tool['examples'] else {}
    pairs.append({
        'instruction': user_q,
        'output': tool_call(tool_name, args),
        'source': f'fc_select:{tool_name}',
        'task_type': 'function_calling',
        'score': 1.0,
    })

print(f'Tool selection: {len(TOOL_SELECTION)}')
print(f'Total: {len(pairs)} function calling pairs')


## Prompt & output variations

Use the pre-warmed base model to generate 9 natural rephrasings of each
unique user-facing instruction. Each variation is paired with a contextual
output that wraps the `<tool_call>` in a natural assistant response, teaching
the model to embed tool calls in conversational flow.

In [ ]:
# ── Prompt & output variations ───────────────────────────────────────────────
# Use the pre-warmed model to generate 9 natural rephrasings of each unique
# user-facing instruction.  Pair each variation with a contextual output that
# embeds the tool call in a natural assistant response.
# ─────────────────────────────────────────────────────────────────────────────

import gc

# ── Load pre-warmed model ────────────────────────────────────────────────────
model, tokenizer, _load, generate, make_sampler = load_model()
sampler = make_sampler(temp=0.7, top_p=0.9)

# ── Output context templates ────────────────────────────────────────────────
# Each tool gets 9 natural lead-in phrases to wrap around the <tool_call>.
TOOL_CONTEXTS = {
    'read_file': [
        "Let me read that file.",
        "I'll pull up the file contents.",
        "Sure, let me take a look.",
        "Let me check what's in there.",
        "I'll open that up for you.",
        "Looking at the file now.",
        "Let me examine that.",
        "I'll read the file to see what we're working with.",
        "Sure, reading the file now.",
    ],
    'write_file': [
        "I'll create that file for you.",
        "Let me write that.",
        "Sure, here's a basic ARO application.",
        "I'll set that up now.",
        "Let me create the file.",
        "Writing the file now.",
        "I'll generate that for you.",
        "Let me put that together.",
        "Creating the file now.",
    ],
    'edit_file': [
        "I'll fix that for you.",
        "Let me apply the edit.",
        "Sure, I'll make that change.",
        "Applying the fix now.",
        "I see the issue — let me correct it.",
        "Let me update that.",
        "I'll patch that up.",
        "Making the edit now.",
        "Sure, here's the fix.",
    ],
    'list_dir': [
        "Let me list the project files.",
        "I'll show you what's in the directory.",
        "Sure, here's the file listing.",
        "Let me check the project structure.",
        "Looking at the directory contents.",
        "I'll see what files are here.",
        "Here's what's in the project.",
        "Let me scan the directory.",
        "Checking the file structure.",
    ],
    'grep': [
        "Let me search for that.",
        "I'll grep through the codebase.",
        "Searching the files now.",
        "Let me find that for you.",
        "I'll look through the source files.",
        "Searching for matches.",
        "Let me scan the code.",
        "I'll search the ARO files.",
        "Looking for that pattern now.",
    ],
    'run_shell': [
        "I'll run that command.",
        "Let me execute that.",
        "Sure, running the command now.",
        "I'll fire that off.",
        "Executing the command.",
        "Let me run that for you.",
        "Sure, here goes.",
        "Running the shell command.",
        "I'll execute that now.",
    ],
    'aro_check': [
        "Let me validate the code.",
        "I'll check for syntax errors.",
        "Running the syntax checker.",
        "Let me verify the code is correct.",
        "I'll run aro check on the project.",
        "Checking the code for errors.",
        "Let me lint the ARO code.",
        "Validating the syntax now.",
        "I'll scan for issues.",
    ],
    'aro_run': [
        "I'll run the application.",
        "Let me execute the app.",
        "Starting the application now.",
        "Running the ARO app.",
        "I'll fire up the application.",
        "Let me start it up.",
        "Executing the application.",
        "I'll launch the app.",
        "Running it now.",
    ],
    'aro_test': [
        "I'll run the tests.",
        "Let me execute the test suite.",
        "Running tests now.",
        "I'll check if the tests pass.",
        "Let me run the test suite.",
        "Executing the tests.",
        "I'll verify with the tests.",
        "Running the test suite.",
        "Let me see if the tests pass.",
    ],
    'aro_build': [
        "I'll build the native binary.",
        "Let me compile the application.",
        "Building the binary now.",
        "I'll compile it for you.",
        "Let me create the native build.",
        "Compiling the application.",
        "I'll generate the binary.",
        "Building the project.",
        "Let me build that.",
    ],
    'parse_aro': [
        "I'll parse the file and show the AST.",
        "Let me generate the parse tree.",
        "Parsing the file now.",
        "I'll show you the abstract syntax tree.",
        "Let me analyze the structure.",
        "Generating the AST.",
        "I'll parse that for you.",
        "Looking at the parse tree.",
        "Let me show the syntax tree.",
    ],
    'list_actions': [
        "I'll list all available actions.",
        "Let me show you what actions exist.",
        "Here are the available ARO actions.",
        "I'll pull up the action reference.",
        "Let me check the available actions.",
        "Listing all actions.",
        "I'll show the action catalog.",
        "Here's what you can use.",
        "Let me look up the actions.",
    ],
    'create_plugin': [
        "I'll scaffold the plugin for you.",
        "Let me create the plugin structure.",
        "Setting up the plugin now.",
        "I'll generate the plugin boilerplate.",
        "Creating the plugin.",
        "Let me set that up.",
        "I'll scaffold that plugin.",
        "Generating the plugin structure.",
        "Let me create the plugin project.",
    ],
    'write_openapi': [
        "I'll generate the OpenAPI contract.",
        "Let me create the API specification.",
        "Writing the openapi.yaml.",
        "I'll draft the API contract.",
        "Creating the OpenAPI spec.",
        "Let me write the contract.",
        "I'll generate the specification.",
        "Setting up the API contract.",
        "Let me create the openapi.yaml.",
    ],
    'generate_docs': [
        "I'll generate the documentation.",
        "Let me create the docs.",
        "Generating documentation now.",
        "I'll build the documentation.",
        "Creating the docs.",
        "Let me generate the docs.",
        "I'll produce the documentation.",
        "Generating the docs now.",
        "Let me document the project.",
    ],
    'list_proposals': [
        "I'll list the language proposals.",
        "Let me show you the proposals.",
        "Here are the ARO proposals.",
        "I'll pull up the proposal list.",
        "Listing the proposals.",
        "Let me check what proposals exist.",
        "I'll show the specifications.",
        "Here's the proposal catalog.",
        "Let me look up the proposals.",
    ],
    'read_proposal': [
        "I'll pull up that proposal.",
        "Let me read the proposal.",
        "Loading the proposal now.",
        "I'll show you that specification.",
        "Reading the proposal.",
        "Let me fetch that proposal.",
        "I'll display the proposal.",
        "Here's the proposal content.",
        "Let me look that up.",
    ],
    'search_project': [
        "I'll search the project for that.",
        "Let me look through the codebase.",
        "Searching the project now.",
        "I'll find relevant code.",
        "Let me search for related files.",
        "Searching for matches.",
        "I'll scan the project.",
        "Let me find that in the codebase.",
        "Looking through the project.",
    ],
}
GENERIC_CONTEXTS = [
    "Sure, let me handle that.",
    "I'll take care of that.",
    "On it.",
    "Let me do that for you.",
    "Sure, here you go.",
    "Working on that now.",
    "I'll handle that.",
    "Let me get that done.",
    "Sure, doing that now.",
]

# ── Filter: only variate user-facing instructions ────────────────────────────
def should_variate(instruction):
    """Skip model-internal prompts (contain code blocks or error traces)."""
    return ('```' not in instruction
            and not ('error:' in instruction.lower() and '\n' in instruction))

variatable_instrs = [p['instruction'] for p in pairs if should_variate(p['instruction'])]
unique_instructions = sorted(set(variatable_instrs))
skipped = len(set(p['instruction'] for p in pairs)) - len(unique_instructions)
print(f'Generating 9 variations for {len(unique_instructions)} unique instructions '
      f'(skipping {skipped} model-internal prompts)...\n')

variation_map = {}
for i, instr in enumerate(unique_instructions):
    prompt_text = (
        "/no_think\n"
        "Rephrase this user request in 9 different ways. Each should sound "
        "natural and express the same intent. Vary the style: questions, "
        "commands, casual, formal, short, and long forms. Preserve any "
        "specific filenames, paths, or technical terms.\n\n"
        f'Original: "{instr}"\n\n'
        "Return ONLY 9 variations, one per line, numbered 1-9."
    )
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    response = generate(model, tokenizer, prompt=formatted,
                        sampler=sampler, max_tokens=600)
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

    variations = []
    for line in response.split('\n'):
        cleaned = re.sub(r'^\d+[\.\)\-]\s*', '', line.strip())
        cleaned = cleaned.strip('"').strip("'").strip('*').strip()
        if cleaned and cleaned != instr and 5 < len(cleaned) < 300:
            variations.append(cleaned)
    variation_map[instr] = variations[:9]

    if (i + 1) % 10 == 0 or i + 1 == len(unique_instructions):
        print(f'  {i+1}/{len(unique_instructions)}')

total_vars = sum(len(v) for v in variation_map.values())
print(f'\nGenerated {total_vars} variations '
      f'(avg {total_vars / max(len(variation_map), 1):.1f} per instruction)')

# ── Build varied pairs with contextual outputs ──────────────────────────────
seen = set((p['instruction'], p['output']) for p in pairs)
varied_pairs = []

for p in pairs:
    instr = p['instruction']
    if not should_variate(instr):
        continue
    original_output = p['output']
    source = p.get('source', '')

    # Resolve tool name for context lookup
    tool_name = source.split(':')[1] if ':' in source else ''
    if tool_name not in TOOL_CONTEXTS:
        m = re.search(r'"name":\s*"(\w+)"', original_output)
        if m:
            tool_name = m.group(1)

    contexts = TOOL_CONTEXTS.get(tool_name, GENERIC_CONTEXTS)
    variations = variation_map.get(instr, [])

    for j, var_instr in enumerate(variations):
        ctx = contexts[j % len(contexts)]
        varied_output = f"{ctx}\n\n{original_output}"
        key = (var_instr, varied_output)
        if key in seen:
            continue
        seen.add(key)
        varied_pairs.append({
            'instruction': var_instr,
            'output': varied_output,
            'source': f'{source}:var{j}',
            'task_type': 'function_calling',
            'score': 1.0,
        })

pairs.extend(varied_pairs)
print(f'Added {len(varied_pairs)} varied pairs → total: {len(pairs)}')

# Semantic alignment gate (audit fix): drop pairs whose code
# doesn't address the instruction. Same judge as NB10. Must run
# BEFORE the model is freed below.
from mlx_lm import generate as _mlx_generate_for_judge
pairs, _gate_dropped = semantic_alignment_filter(
    pairs, model, tokenizer, _mlx_generate_for_judge,
    make_sampler, label='NB11',
)
print(f'semantic gate: dropped {_gate_dropped} pairs, '
      f'{len(pairs)} remaining')

# ── Free model memory ────────────────────────────────────────────────────────
import mlx.core as mx
del model, tokenizer
gc.collect(); mx.metal.clear_cache()
print('Model unloaded.')

## Error-recovery fix chains & tool-failure traces

Two additional trace families, both stored as full multi-turn conversations
(`task_type='tool_calling'` so NB16 keeps the message list intact):

1. **Extended fix chains** (issue #398) — 8–12 step debugging sessions where the
   *first* fix attempt does not resolve the problem: re-checking surfaces a
   *different* error, the assistant re-diagnoses the root cause, applies an
   alternative fix, and only then verification succeeds. Reasoning text is
   interleaved between tool calls.
2. **Tool-failure recovery** (issue #397) — traces where a tool call itself
   fails (`old_string not found`, `File not found`, non-unique `old_string`)
   and the assistant recovers by gathering more context and retrying, instead
   of giving up or hallucinating a result.

In [ ]:
# ── Extended fix chains: first fix fails → re-diagnose → alternative fix ─────
# (issue #398) Happy-path read→check→edit→verify chains teach the mechanics,
# but real debugging sessions usually need more than one attempt. Each
# scenario below has TWO independent bugs: fixing the first surfaces a
# different error on re-check, forcing re-diagnosis before success.

def _tool_result(name, text):
    return f'Tool result ({name}):\n{text}'

EXTENDED_FIX_SCENARIOS = [
    {
        'id': 'concat_then_rebind',
        'request': 'aro check fails on my project. Please fix main.aro.',
        'code': ('(Application-Start: Greeter) {\n'
                 '    Create the <name> with "World".\n'
                 '    Compute the <name> from "Hello, " + <name>.\n'
                 '    Log <name> to the <console>.\n'
                 '    Return an <OK: status> for the <greeting>.\n'
                 '}'),
        'error1': "main.aro:3:38: error: Expected '>', but got +",
        'diagnosis1': ('The error points at the `+` in line 3. ARO uses `++` for string '
                       'concatenation — `+` is arithmetic only. Let me fix that.'),
        'edit1': {'old': '"Hello, " + <name>', 'new': '"Hello, " ++ <name>'},
        'error2': "main.aro:3:5: error: Cannot rebind variable 'name'",
        'diagnosis2': ('The concatenation fix was correct, but re-checking reveals a second, '
                       'unrelated problem: line 3 rebinds `<name>`, and ARO variables are '
                       'immutable — each transformation needs a NEW name. I will rename the '
                       'computed value to `<greeting-message>` and update the Log statement '
                       'that reads it.'),
        'edit2': {'old': ('Compute the <name> from "Hello, " ++ <name>.\n'
                          '    Log <name> to the <console>.'),
                  'new': ('Compute the <greeting-message> from "Hello, " ++ <name>.\n'
                          '    Log <greeting-message> to the <console>.')},
        'summary': ('Fixed — there were two separate bugs:\n'
                    '1. String concatenation used `+` instead of `++`.\n'
                    '2. The result was rebound to the immutable variable `<name>`; it now '
                    'uses a new name, `<greeting-message>`.\n'
                    '`aro check` passes.'),
    },
    {
        'id': 'store_then_retrieve_preposition',
        'request': 'My user service will not compile. Can you debug users.aro?',
        'file': 'users.aro',
        'code': ('(createUser: User API) {\n'
                 '    Extract the <data> from the <request: body>.\n'
                 '    Create the <user> with <data>.\n'
                 '    Store the <user> from the <user-repository>.\n'
                 '    Return a <Created: status> with <user>.\n'
                 '}\n'
                 '\n'
                 '(getUser: User API) {\n'
                 '    Extract the <id> from the <pathParameters: id>.\n'
                 '    Retrieve the <user> with the <user-repository> where id = <id>.\n'
                 '    Return an <OK: status> with <user>.\n'
                 '}'),
        'error1': "users.aro:4:22: error: Expected preposition 'into', but got 'from'",
        'diagnosis1': ('Line 4 stores a user but uses `from` — `Store` writes INTO a '
                       'repository, so the preposition must be `into`. Fixing that first.'),
        'edit1': {'old': 'Store the <user> from the <user-repository>.',
                  'new': 'Store the <user> into the <user-repository>.'},
        'error2': "users.aro:10:25: error: Expected preposition 'from', but got 'with'",
        'diagnosis2': ('The Store fix worked, but the checker now reaches getUser and finds '
                       'the mirror-image mistake: `Retrieve` READS from a repository, so it '
                       'needs `from`, not `with`. Applying the second fix.'),
        'edit2': {'old': 'Retrieve the <user> with the <user-repository> where id = <id>.',
                  'new': 'Retrieve the <user> from the <user-repository> where id = <id>.'},
        'summary': ('Both endpoints compile now. The repository prepositions were swapped:\n'
                    '- `Store` writes, so it takes `into the <user-repository>`.\n'
                    '- `Retrieve` reads, so it takes `from the <user-repository>`.\n'
                    'Rule of thumb: data flows in the direction of the preposition.'),
    },
    {
        'id': 'emit_then_missing_start',
        'request': 'Something is wrong with my order app — aro check keeps failing.',
        'code': ('(placeOrder: Order API) {\n'
                 '    Extract the <order> from the <request: body>.\n'
                 '    Store the <order> into the <order-repository>.\n'
                 '    Emit the <order> to the <order-placed-event>.\n'
                 '    Return a <Created: status> with <order>.\n'
                 '}'),
        'error1': 'main.aro:4:10: error: Expected event qualifier',
        'diagnosis1': ('The Emit statement is inverted: the event type belongs in result '
                       'position with an `: event` qualifier, and the payload follows '
                       '`with`. Rewriting it as `Emit an <OrderPlaced: event> with <order>.`'),
        'edit1': {'old': 'Emit the <order> to the <order-placed-event>.',
                  'new': 'Emit an <OrderPlaced: event> with <order>.'},
        'error2': 'error: Entry point not found — every application needs exactly one Application-Start feature set',
        'diagnosis2': ('The Emit syntax is fixed, but the re-check surfaces a structural '
                       'problem the parser could not report before: the application has no '
                       '`Application-Start` feature set at all. I will add one that starts '
                       'the HTTP server and keeps the app alive.'),
        'edit2': {'old': '(placeOrder: Order API) {',
                  'new': ('(Application-Start: Order Service) {\n'
                          '    Start the <http-server> with <contract>.\n'
                          '    Keepalive the <application> for the <events>.\n'
                          '    Return an <OK: status> for the <startup>.\n'
                          '}\n'
                          '\n'
                          '(placeOrder: Order API) {')},
        'summary': ('Fixed in two steps:\n'
                    '1. `Emit` takes the event type in result position — '
                    '`Emit an <OrderPlaced: event> with <order>.`\n'
                    '2. The app was missing its required `Application-Start` feature set; '
                    'I added one that starts the HTTP server with the contract and calls '
                    'Keepalive so the server stays up.'),
    },
    {
        'id': 'reserved_prefix_then_log_preposition',
        'request': 'Please fix the validation feature in main.aro, aro check rejects it.',
        'code': ('(Application-Start: Validator) {\n'
                 '    Create the <age> with 25.\n'
                 '    Compute the <is-adult> from <age> > 18.\n'
                 '    Log <is-adult> for the <console>.\n'
                 '    Return an <OK: status> for the <validation>.\n'
                 '}'),
        'error1': "main.aro:3:18: error: Reserved prefix 'is-'",
        'diagnosis1': ('`is-` is a reserved variable-name prefix in ARO. I will rename '
                       '`<is-adult>` to `<adult>` in both places it appears.'),
        'edit1': {'old': ('Compute the <is-adult> from <age> > 18.\n'
                          '    Log <is-adult> for the <console>.'),
                  'new': ('Compute the <adult> from <age> > 18.\n'
                          '    Log <adult> for the <console>.')},
        'error2': "main.aro:4:16: error: Expected preposition 'to', but got 'for'",
        'diagnosis2': ('The rename was necessary but not sufficient — the Log statement '
                       'also uses the wrong preposition. `Log` writes TO a sink: '
                       '`Log <adult> to the <console>.`'),
        'edit2': {'old': 'Log <adult> for the <console>.',
                  'new': 'Log <adult> to the <console>.'},
        'summary': ('Two fixes were needed:\n'
                    '1. `<is-adult>` used the reserved `is-` prefix → renamed to `<adult>`.\n'
                    '2. `Log ... for the <console>` used the wrong preposition → `Log ... to '
                    'the <console>`.\n'
                    'The feature set now passes `aro check`.'),
    },
]

extended_pairs = []
for s in EXTENDED_FIX_SCENARIOS:
    fname = s.get('file', 'main.aro')
    numbered = '\n'.join(f'{i+1}\t{l}' for i, l in enumerate(s['code'].split('\n')))
    msgs = [
        {'role': 'user', 'content': s['request']},
        {'role': 'assistant', 'content':
            f"Let me look at {fname} first to see what we're working with.\n\n"
            + tool_call('read_file', {'path': fname})},
        {'role': 'user', 'content': _tool_result('read_file', numbered)},
        {'role': 'assistant', 'content':
            'Now let me run the syntax checker to get the exact error.\n\n'
            + tool_call('aro_check', {'path': '.'})},
        {'role': 'user', 'content': _tool_result('aro_check', s['error1'])},
        {'role': 'assistant', 'content':
            s['diagnosis1'] + '\n\n'
            + tool_call('edit_file', {'path': fname,
                                      'old_string': s['edit1']['old'],
                                      'new_string': s['edit1']['new']})},
        {'role': 'user', 'content': _tool_result('edit_file', f'Applied edit to {fname}')},
        {'role': 'assistant', 'content':
            'Let me verify that fix resolved the problem.\n\n'
            + tool_call('aro_check', {'path': '.'})},
        {'role': 'user', 'content': _tool_result('aro_check', s['error2'])},
        {'role': 'assistant', 'content':
            'That is a different error — the first fix was correct but incomplete. '
            + s['diagnosis2'] + '\n\n'
            + tool_call('edit_file', {'path': fname,
                                      'old_string': s['edit2']['old'],
                                      'new_string': s['edit2']['new']})},
        {'role': 'user', 'content': _tool_result('edit_file', f'Applied edit to {fname}')},
        {'role': 'assistant', 'content':
            'Checking once more to confirm everything passes now.\n\n'
            + tool_call('aro_check', {'path': '.'})},
        {'role': 'user', 'content': _tool_result('aro_check', 'No errors found. 1 file checked.')},
        {'role': 'assistant', 'content': s['summary']},
    ]
    extended_pairs.append({
        'messages':  msgs,
        'source':    f"fc_fix_chain:{s['id']}",
        'task_type': 'tool_calling',
        'score':     1.0,
    })

pairs.extend(extended_pairs)
print(f'Extended fix chains: {len(extended_pairs)} traces '
      f'({len(EXTENDED_FIX_SCENARIOS[0]["code"].splitlines())}-line programs, '
      f'{len(extended_pairs[0]["messages"])} messages each)')


In [ ]:
# ── Tool-failure recovery traces (issue #397) ────────────────────────────────
# Teach the model what REAL tool errors look like (formats match AskToolError
# in Sources/AROAsk/Tools/AskTool.swift) and how to recover: gather more
# context, then retry — never invent a result or silently give up.

TOOL_FAILURE_TRACES = []

# 1. edit_file → old_string not found → re-read the file → retry with exact text
TOOL_FAILURE_TRACES.append({
    'source': 'fc_tool_failure:edit_not_found',
    'messages': [
        {'role': 'user', 'content': 'Change the log message in main.aro to say "Server ready".'},
        {'role': 'assistant', 'content':
            "I'll update the log statement.\n\n"
            + tool_call('edit_file', {'path': 'main.aro',
                                      'old_string': 'Log "Starting..." to the <console>.',
                                      'new_string': 'Log "Server ready" to the <console>.'})},
        {'role': 'user', 'content': _tool_result('edit_file',
            'Tool execution failed: old_string not found in main.aro')},
        {'role': 'assistant', 'content':
            'The exact text I tried to replace is not in the file. Let me read '
            'main.aro to see the actual log statement instead of guessing.\n\n'
            + tool_call('read_file', {'path': 'main.aro'})},
        {'role': 'user', 'content': _tool_result('read_file',
            '1\t(Application-Start: My API) {\n'
            '2\t    Log "Booting up" to the <console>.\n'
            '3\t    Start the <http-server> with <contract>.\n'
            '4\t    Keepalive the <application> for the <events>.\n'
            '5\t    Return an <OK: status> for the <startup>.\n'
            '6\t}')},
        {'role': 'assistant', 'content':
            'The actual message is "Booting up", not "Starting...". Retrying '
            'with the exact text from the file.\n\n'
            + tool_call('edit_file', {'path': 'main.aro',
                                      'old_string': 'Log "Booting up" to the <console>.',
                                      'new_string': 'Log "Server ready" to the <console>.'})},
        {'role': 'user', 'content': _tool_result('edit_file', 'Applied edit to main.aro')},
        {'role': 'assistant', 'content':
            'Done — the startup log now reads "Server ready". The first edit failed '
            'because I assumed the old text instead of reading the file; always match '
            'the exact current content when editing.'},
    ],
})

# 2. read_file → File not found → list_dir to discover the real name → read it
TOOL_FAILURE_TRACES.append({
    'source': 'fc_tool_failure:file_not_found',
    'messages': [
        {'role': 'user', 'content': 'Show me the user endpoints in user.aro.'},
        {'role': 'assistant', 'content':
            'Let me open that file.\n\n' + tool_call('read_file', {'path': 'user.aro'})},
        {'role': 'user', 'content': _tool_result('read_file',
            'Tool execution failed: File not found: user.aro')},
        {'role': 'assistant', 'content':
            "There is no user.aro — the file may be named differently. Let me list "
            'the project to find it.\n\n' + tool_call('list_dir', {})},
        {'role': 'user', 'content': _tool_result('list_dir',
            'main.aro\nopenapi.yaml\nusers.aro\norders.aro')},
        {'role': 'assistant', 'content':
            'The file is called users.aro (plural). Reading that instead.\n\n'
            + tool_call('read_file', {'path': 'users.aro'})},
        {'role': 'user', 'content': _tool_result('read_file',
            '1\t(listUsers: User API) {\n'
            '2\t    Retrieve the <users> from the <user-repository>.\n'
            '3\t    Return an <OK: status> with <users>.\n'
            '4\t}')},
        {'role': 'assistant', 'content':
            'Found it — users.aro defines one endpoint:\n\n'
            '- **listUsers** (`User API`): retrieves all users from the '
            '`<user-repository>` and returns them with an `<OK: status>`.'},
    ],
})

# 3. edit_file → old_string not unique → retry with more surrounding context
TOOL_FAILURE_TRACES.append({
    'source': 'fc_tool_failure:edit_not_unique',
    'messages': [
        {'role': 'user', 'content':
            'In orders.aro, the getOrder endpoint should return NotFound context — '
            'change its Return statement.'},
        {'role': 'assistant', 'content':
            "I'll edit the Return statement in getOrder.\n\n"
            + tool_call('edit_file', {'path': 'orders.aro',
                                      'old_string': 'Return an <OK: status> with <order>.',
                                      'new_string': 'Return an <OK: status> with <order> for the <lookup>.'})},
        {'role': 'user', 'content': _tool_result('edit_file',
            'Tool execution failed: old_string appears 3 times in orders.aro — '
            'must be unique. Provide more surrounding context.')},
        {'role': 'assistant', 'content':
            'That line appears in three endpoints, so the edit is ambiguous. I need '
            'to include the surrounding getOrder-specific lines to make the match '
            'unique.\n\n'
            + tool_call('edit_file', {'path': 'orders.aro',
                'old_string': ('Retrieve the <order> from the <order-repository> where id = <id>.\n'
                               '    Return an <OK: status> with <order>.'),
                'new_string': ('Retrieve the <order> from the <order-repository> where id = <id>.\n'
                               '    Return an <OK: status> with <order> for the <lookup>.')})},
        {'role': 'user', 'content': _tool_result('edit_file', 'Applied edit to orders.aro')},
        {'role': 'assistant', 'content':
            'Edit applied. Because the Return line existed in three feature sets, I '
            'anchored the replacement to the unique Retrieve line directly above it '
            'in getOrder.'},
    ],
})

# 4. aro_run fails at runtime → read → fix → re-run success
TOOL_FAILURE_TRACES.append({
    'source': 'fc_tool_failure:run_crash_recovery',
    'messages': [
        {'role': 'user', 'content': 'aro check passes but the app crashes when I run it. Help!'},
        {'role': 'assistant', 'content':
            'Let me reproduce the crash first.\n\n' + tool_call('aro_run', {'path': '.'})},
        {'role': 'user', 'content': _tool_result('aro_run',
            'Can not retrieve the user from the user-repository where id = 42\nFailed (exit 1).')},
        {'role': 'assistant', 'content':
            'The runtime error says the repository lookup found nothing — the app '
            'reads a user that was never stored. Let me check main.aro.\n\n'
            + tool_call('read_file', {'path': 'main.aro'})},
        {'role': 'user', 'content': _tool_result('read_file',
            '1\t(Application-Start: Demo) {\n'
            '2\t    Retrieve the <user> from the <user-repository> where id = 42.\n'
            '3\t    Log <user> to the <console>.\n'
            '4\t    Return an <OK: status> for the <startup>.\n'
            '5\t}')},
        {'role': 'assistant', 'content':
            'Confirmed: the repository is empty at startup, so the Retrieve fails at '
            'runtime even though the syntax is valid. The fix is to store a user '
            'before retrieving it.\n\n'
            + tool_call('edit_file', {'path': 'main.aro',
                'old_string': 'Retrieve the <user> from the <user-repository> where id = 42.',
                'new_string': ('Create the <new-user> with { id: 42, name: "Ada" }.\n'
                               '    Store the <new-user> into the <user-repository>.\n'
                               '    Retrieve the <user> from the <user-repository> where id = 42.')})},
        {'role': 'user', 'content': _tool_result('edit_file', 'Applied edit to main.aro')},
        {'role': 'assistant', 'content':
            'Running it again to confirm.\n\n' + tool_call('aro_run', {'path': '.'})},
        {'role': 'user', 'content': _tool_result('aro_run',
            '{ id: 42, name: "Ada" }\nCompleted successfully.')},
        {'role': 'assistant', 'content':
            'The app runs cleanly now. `aro check` only validates syntax — this was a '
            'runtime error: retrieving from an empty repository. Seeding the '
            '`<user-repository>` with a Store before the Retrieve fixed it.'},
    ],
})

failure_pairs = [{
    'messages':  t['messages'],
    'source':    t['source'],
    'task_type': 'tool_calling',
    'score':     1.0,
} for t in TOOL_FAILURE_TRACES]

pairs.extend(failure_pairs)
print(f'Tool-failure recovery traces: {len(failure_pairs)}')
print(f'Total pairs: {len(pairs)}')


## OPEN FILE (editor focus) & write-into-file workflow traces

`aro ask --file <path>` and SOLARO's co-pilot inject the currently open
editor file as a transient "OPEN FILE: <path>" system block (AskSession
.focusFileMessage). These traces teach the runtime contract:

1. The block **is** the file content — no `read_file` round-trip.
2. Unnamed requests ("this file", "fix this", "add X here") target the open file.
3. WRITE/CREATE requests end in `write_file`/`edit_file` + `aro_check` +
   a **short summary** — never a full code dump into the chat.
4. An explicit path in the request overrides the focus file.

The leading placeholder system message is replaced with the canonical
system prompt by NB16 `to_chatml`; the OPEN FILE block is the *second*
system message, mirroring the runtime injection order exactly.


In [ ]:
# ── OPEN FILE (editor focus) + write-into-file workflow traces ───────────────
# Mirrors AskSession.focusFileMessage() in Sources/AROAsk/AskSession.swift:
# the block text below must stay in sync with the runtime injection.

# NB16 to_chatml() replaces messages[0] (role=system) with the canonical
# system prompt, so every trace leads with this placeholder and puts the
# OPEN FILE block SECOND — same order the runtime produces.
_SYSTEM_PLACEHOLDER = {'role': 'system', 'content':
    '(canonical ARO system prompt — replaced at dataset assembly)'}

def _open_file_block(path, content):
    return {'role': 'system', 'content':
        f'OPEN FILE: {path}\n'
        'This is the file the user has open in the editor right now. It is '
        'the default target: when the user says "this file", "this code", or '
        'asks for a change without naming a file, they mean this one. Its '
        'current content is below — you do not need read_file for it. Apply '
        f'modifications with edit_file or write_file using the path "{path}".\n'
        f'```\n{content}\n```'}

OPEN_FILE_TRACES = []

# 1. Unnamed edit request → edit the open file directly, no read_file.
OPEN_FILE_TRACES.append({
    'source': 'fc_open_file:edit_direct',
    'messages': [
        _SYSTEM_PLACEHOLDER,
        _open_file_block('greeter.aro',
            '(Application-Start: Greeter) {\n'
            '    Create the <name> with "World".\n'
            '    Compute the <greeting> from "Hello, " ++ <name>.\n'
            '    Return an <OK: status> for the <startup>.\n'
            '}'),
        {'role': 'user', 'content': 'Add a log line to this file so the greeting is printed on startup.'},
        {'role': 'assistant', 'content':
            'The open file greeter.aro computes the greeting but never logs it. '
            'I will insert a Log statement before the Return.\n\n'
            + tool_call('edit_file', {'path': 'greeter.aro',
                'old_string': '    Return an <OK: status> for the <startup>.',
                'new_string': '    Log <greeting> to the <console>.\n'
                              '    Return an <OK: status> for the <startup>.'})},
        {'role': 'user', 'content': _tool_result('edit_file', 'Applied edit to greeter.aro')},
        {'role': 'assistant', 'content':
            'Verifying the change.\n\n' + tool_call('aro_check', {'path': '.'})},
        {'role': 'user', 'content': _tool_result('aro_check', 'Check passed: 1 file, 1 feature set. No errors.')},
        {'role': 'assistant', 'content':
            'Done — greeter.aro now logs the greeting to the console before '
            'returning. `aro check` passes.'},
    ],
})

# 2. Feature request against the open file → append a feature set in place.
OPEN_FILE_TRACES.append({
    'source': 'fc_open_file:add_endpoint',
    'messages': [
        _SYSTEM_PLACEHOLDER,
        _open_file_block('users.aro',
            '(listUsers: User API) {\n'
            '    Retrieve the <users> from the <user-repository>.\n'
            '    Return an <OK: status> with <users>.\n'
            '}\n'
            '\n'
            '(getUser: User API) {\n'
            '    Extract the <id> from the <pathParameters: id>.\n'
            '    Retrieve the <user> from the <user-repository> where id = <id>.\n'
            '    Return an <OK: status> with <user>.\n'
            '}'),
        {'role': 'user', 'content': 'Add an endpoint that deletes a user by id.'},
        {'role': 'assistant', 'content':
            'I will add a deleteUser feature set to the open file users.aro, '
            'following the same pattern as getUser.\n\n'
            + tool_call('edit_file', {'path': 'users.aro',
                'old_string': '    Retrieve the <user> from the <user-repository> where id = <id>.\n'
                              '    Return an <OK: status> with <user>.\n'
                              '}',
                'new_string': '    Retrieve the <user> from the <user-repository> where id = <id>.\n'
                              '    Return an <OK: status> with <user>.\n'
                              '}\n'
                              '\n'
                              '(deleteUser: User API) {\n'
                              '    Extract the <id> from the <pathParameters: id>.\n'
                              '    Delete the <user> from the <user-repository> where id = <id>.\n'
                              '    Return an <OK: status> for the <deletion>.\n'
                              '}'})},
        {'role': 'user', 'content': _tool_result('edit_file', 'Applied edit to users.aro')},
        {'role': 'assistant', 'content':
            'Checking syntax.\n\n' + tool_call('aro_check', {'path': '.'})},
        {'role': 'user', 'content': _tool_result('aro_check', 'Check passed: 1 file, 3 feature sets. No errors.')},
        {'role': 'assistant', 'content':
            'Added `deleteUser` to users.aro — it extracts the id from the path '
            'parameters, deletes the matching user, and returns OK. Remember to '
            'declare the route in openapi.yaml with `operationId: deleteUser` '
            '(e.g. DELETE /users/{id}) or the HTTP server will not dispatch to it.'},
    ],
})

# 3. Explain request → answer from the OPEN FILE block, zero tool calls.
OPEN_FILE_TRACES.append({
    'source': 'fc_open_file:explain_no_tools',
    'messages': [
        _SYSTEM_PLACEHOLDER,
        _open_file_block('pipeline.aro',
            '(Application-Start: Sales Report) {\n'
            '    Read the <orders> from the <file: "orders.json">.\n'
            '    Filter the <large-orders> from the <orders> where total > 100.\n'
            '    Group the <by-region> from the <large-orders> by region.\n'
            '    Log <by-region> to the <console>.\n'
            '    Return an <OK: status> for the <report>.\n'
            '}'),
        {'role': 'user', 'content': 'What does this code do?'},
        {'role': 'assistant', 'content':
            'This is a small data-pipeline application (the open file, '
            'pipeline.aro):\n\n'
            '1. **Read** loads `orders.json` into `<orders>` — the format is '
            'detected automatically.\n'
            '2. **Filter** keeps only orders with a `total` above 100 as '
            '`<large-orders>`.\n'
            '3. **Group** partitions those orders by their `region` field into '
            '`<by-region>` — a map of region name to order list.\n'
            '4. **Log** prints the grouped result to the console, and the feature '
            'set ends with the mandatory Return.\n\n'
            'Because every action is lazy, nothing is read or filtered until the '
            'Log forces the pipeline; with a huge orders file only the rows that '
            'survive the filter are materialised.'},
    ],
})

# 4. No open file, WRITE request → write_file + aro_check + short summary
#    (source goes into the file, not the chat).
OPEN_FILE_TRACES.append({
    'source': 'fc_write_workflow:new_app',
    'messages': [
        _SYSTEM_PLACEHOLDER,
        {'role': 'user', 'content': 'Create a small ARO app that converts 25 degrees Celsius to Fahrenheit and logs the result.'},
        {'role': 'assistant', 'content':
            'I will create main.aro with the conversion.\n\n'
            + tool_call('write_file', {'path': 'main.aro',
                'content': '(Application-Start: Temperature Converter) {\n'
                           '    Create the <celsius> with 25.\n'
                           '    Compute the <scaled> from <celsius> * 9 / 5.\n'
                           '    Compute the <fahrenheit> from <scaled> + 32.\n'
                           '    Log <fahrenheit> to the <console>.\n'
                           '    Return an <OK: status> for the <conversion>.\n'
                           '}'})},
        {'role': 'user', 'content': _tool_result('write_file', 'Wrote 262 bytes to main.aro')},
        {'role': 'assistant', 'content':
            'Validating.\n\n' + tool_call('aro_check', {'path': '.'})},
        {'role': 'user', 'content': _tool_result('aro_check', 'Check passed: 1 file, 1 feature set. No errors.')},
        {'role': 'assistant', 'content':
            'Created main.aro — it converts 25 °C to Fahrenheit in two Compute '
            'steps (immutability means each intermediate gets its own name) and '
            'logs 77. `aro check` passes; run it with `aro run .`.'},
    ],
})

# 5. "Fix this" against the open file → diagnose from the block, edit, verify.
OPEN_FILE_TRACES.append({
    'source': 'fc_open_file:fix_direct',
    'messages': [
        _SYSTEM_PLACEHOLDER,
        _open_file_block('welcome.aro',
            '(Application-Start: Welcome) {\n'
            '    Create the <user> with "Ada".\n'
            '    Compute the <message> from "Welcome, " + <user>.\n'
            '    Log <message> to the <console>.\n'
            '    Return an <OK: status> for the <startup>.\n'
            '}'),
        {'role': 'user', 'content': 'aro check fails on this file. Fix it.'},
        {'role': 'assistant', 'content':
            'The bug is visible in the open file: line 3 concatenates a string '
            'with `+`, but in ARO `+` is arithmetic only — string concatenation '
            'is `++`. Fixing it directly.\n\n'
            + tool_call('edit_file', {'path': 'welcome.aro',
                'old_string': '"Welcome, " + <user>',
                'new_string': '"Welcome, " ++ <user>'})},
        {'role': 'user', 'content': _tool_result('edit_file', 'Applied edit to welcome.aro')},
        {'role': 'assistant', 'content':
            'Re-checking.\n\n' + tool_call('aro_check', {'path': '.'})},
        {'role': 'user', 'content': _tool_result('aro_check', 'Check passed: 1 file, 1 feature set. No errors.')},
        {'role': 'assistant', 'content':
            'Fixed — the string concatenation in welcome.aro now uses `++` and '
            '`aro check` passes.'},
    ],
})

# 6. Open file present, but the user names a DIFFERENT file → the explicit
#    path wins; read_file the named file instead of assuming the focus.
OPEN_FILE_TRACES.append({
    'source': 'fc_open_file:explicit_path_wins',
    'messages': [
        _SYSTEM_PLACEHOLDER,
        _open_file_block('main.aro',
            '(Application-Start: Shop) {\n'
            '    Start the <http-server> with <contract>.\n'
            '    Keepalive the <application> for the <events>.\n'
            '    Return an <OK: status> for the <startup>.\n'
            '}'),
        {'role': 'user', 'content': 'What feature sets are defined in orders.aro?'},
        {'role': 'assistant', 'content':
            'The user asked about orders.aro, not the open file, so I need to '
            'read it.\n\n' + tool_call('read_file', {'path': 'orders.aro'})},
        {'role': 'user', 'content': _tool_result('read_file',
            '1\t(listOrders: Order API) {\n'
            '2\t    Retrieve the <orders> from the <order-repository>.\n'
            '3\t    Return an <OK: status> with <orders>.\n'
            '4\t}\n'
            '5\t\n'
            '6\t(createOrder: Order API) {\n'
            '7\t    Extract the <data> from the <request: body>.\n'
            '8\t    Create the <order> with <data>.\n'
            '9\t    Emit an <OrderCreated: event> with <order>.\n'
            '10\t    Return a <Created: status> with <order>.\n'
            '11\t}')},
        {'role': 'assistant', 'content':
            'orders.aro defines two HTTP feature sets:\n\n'
            '- **listOrders** — returns every order from the order-repository.\n'
            '- **createOrder** — creates an order from the request body, emits '
            'an `OrderCreated` event for downstream handlers, and returns 201 '
            'Created with the new order.\n\n'
            'Both names must match `operationId` values in openapi.yaml to be '
            'routed.'},
    ],
})

open_file_pairs = [{
    'messages':  t['messages'],
    'source':    t['source'],
    'task_type': 'tool_calling',
    'score':     1.0,
} for t in OPEN_FILE_TRACES]

pairs.extend(open_file_pairs)
print(f'OPEN FILE / write-workflow traces: {len(open_file_pairs)}')
print(f'Total pairs: {len(pairs)}')


## Save and merge

In [ ]:
clean_notebook_pairs('NB11')
for p in pairs:
    p['notebook'] = 'NB11'
new_count = save_notebook_pairs('NB11', pairs)
print(f'Saved {new_count} function calling pairs to {PAIRS_FILE}')

own_file = DATA_OUT / 'function_calling_pairs.jsonl'
with open(own_file, 'w') as f:
    for p in pairs:
        f.write(json.dumps(p) + '\n')
print(f'Local copy: {own_file}')


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white', 'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa', 'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa', 'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '11_function_calling.png'

_tool_counts = Counter()
_type_counts = Counter()
for p in pairs:
    src = p.get('source', '')
    tool = src.split(':')[1] if ':' in src else 'other'
    _tool_counts[tool] += 1
    ptype = src.split(':')[0].replace('fc_', '')
    _type_counts[ptype] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

_sorted = sorted(_tool_counts.items(), key=lambda x: -x[1])
_names = [t for t, _ in _sorted]
_vals = [c for _, c in _sorted]
x = np.arange(len(_names))
ax1.bar(x, _vals, color='#3498db', edgecolor='white', width=0.7)
for i, v in enumerate(_vals):
    ax1.text(i, v + 0.2, str(v), ha='center', fontsize=8, fontweight='bold')
ax1.set_xticks(list(x))
ax1.set_xticklabels(_names, rotation=45, ha='right', fontsize=7)
ax1.set_ylabel('Pairs')
ax1.set_title('Pairs per tool', fontsize=11, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

_ts = sorted(_type_counts.items(), key=lambda x: -x[1])
ax2.bar([t for t,_ in _ts], [c for _,c in _ts],
        color=['#2ecc71','#3498db','#e67e22','#9b59b6'][:len(_ts)], edgecolor='white')
for i, (t, c) in enumerate(_ts):
    ax2.text(i, c + 0.2, str(c), ha='center', fontsize=9, fontweight='bold')
ax2.set_ylabel('Pairs')
ax2.set_title('Pairs by type', fontsize=11, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

fig.suptitle(f'Function Calling Training \u2014 {len(pairs)} pairs, {len(TOOLS)} tools',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')


In [ ]:
import csv
from datetime import date as _date
_run_dir = Path('.') / 'run' / _date.today().isoformat()
csv_path = _run_dir / '11_function_calling.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['tool', 'pair_type', 'instruction_preview', 'output_preview'])
    for p in pairs:
        src = p.get('source', '')
        tool = src.split(':')[1] if ':' in src else 'other'
        ptype = src.split(':')[0].replace('fc_', '')
        if 'messages' in p:
            _user = p['messages'][0]['content'] if p['messages'] else ''
            _asst = p['messages'][-1]['content'] if p['messages'] else ''
            w.writerow([tool, ptype, _user[:150], _asst[:200]])
        else:
            w.writerow([tool, ptype, p['instruction'][:150], p['output'][:200]])
print(f'CSV: {csv_path} ({len(pairs)} rows)')
